## A map, not a maximum

The only campaign here that answers *how many different ways does this go wrong* rather
than *how badly*. Its archive is keyed on the failure mode and on how close the crossing
came, so a cell that collides and a cell that gives up are different entries rather than
two points with the same score.

**What to look for:** how many rows have anything in them. An empty row is a kind of
failure this system does not exhibit — which is a result.


In [ ]:
# DATA_DIR is replaced by the service with the node being viewed. The assignment must stay
# a plain literal for that substitution to work.
DATA_DIR = ''

import json, os, sqlite3
import pandas as pd
import matplotlib.pyplot as plt

def load_units(data_dir):
    """One row per evaluated cell: its parameters, objectives and measures.

    Read from campaign.db rather than data.db because that is where a SEARCH records what
    it scored -- data.db holds per-run tables, and a search's unit of analysis is the cell.
    """
    db = os.path.join(data_dir, 'campaign.db')
    if not os.path.exists(db):
        return pd.DataFrame()
    with sqlite3.connect(db) as conn:
        units = pd.read_sql_query(
            "SELECT u.paramset_id, u.config_name, u.params_json, u.objectives_json,"
            "       u.measures_json, u.n_samples, u.status, b.idx AS batch"
            "  FROM unit u LEFT JOIN batch b ON b.id = u.batch_id"
            " ORDER BY b.idx, u.id", conn)
    if units.empty:
        return units
    for col, prefix in (('params_json', ''), ('objectives_json', ''), ('measures_json', 'm_')):
        expanded = units[col].apply(lambda s: json.loads(s) if s else {}).apply(pd.Series)
        expanded.columns = [f'{prefix}{c}' for c in expanded.columns]
        units = pd.concat([units.drop(columns=[col]), expanded], axis=1)
    return units

units = load_units(DATA_DIR)
scored = units[units['status'] == 'evaluated'] if 'status' in units else units
print(f"{len(units)} cell(s) recorded, {len(scored)} scored")

if scored.empty:
    print("No scored cells yet. A search records a cell once its batch has been evaluated;"
          "\nif this campaign failed early, its controller log says why.")


In [ ]:
TITLE = 'Quality-diversity — how many kinds of trouble'

# The archive as it is keyed: which KIND of trouble, against how close it came. A filled
# square is a kind that actually happens; an empty one is a kind that does not.
if not scored.empty and 'm_failure_mode' in scored:
    modes = ['none', 'collision', 'timeout', 'goal_miss']
    fig, ax = plt.subplots(figsize=(6.5, 4))
    for i, mode in enumerate(modes):
        rows = scored[scored['m_failure_mode'] == mode]
        ax.scatter(rows['m_min_clearance'], [i] * len(rows),
                   s=110, alpha=0.75, edgecolor='black', linewidth=0.4)
    ax.set_yticks(range(len(modes))); ax.set_yticklabels(modes)
    ax.set_xlabel('minimum clearance [m]'); ax.set_ylabel('failure mode')
    ax.axvline(0, color='crimson', linestyle='--', linewidth=1, label='contact')
    ax.set_title('%s: %d distinct kind(s) observed'
                 % (TITLE, scored['m_failure_mode'].nunique()))
    ax.legend(); plt.tight_layout(); plt.show()


In [ ]:
# Computed, never written into the text above: a notebook must not be able to claim a
# finding its own campaign does not support.
if not scored.empty:
    failed = (scored['robustness'] < 0).sum()
    at_ends = ((scored['m_failure_rate'] == 0) | (scored['m_failure_rate'] == 1)).mean() \
        if 'm_failure_rate' in scored else float('nan')
    print(f"cells scored          : {len(scored)}")
    print(f"runs spent            : {int(scored['n_samples'].sum())}")
    print(f"cells that failed     : {failed}  ({failed / len(scored):.0%})")
    print(f"worst robustness      : {scored['robustness'].min():.3f}")
    print(f"closest to the edge   : {scored['robustness'].abs().min():.3f}")
    print()
    print("The comparison this campaign is FOR:")
    print(f"  failure_rate sits at 0 or 1 for {at_ends:.0%} of cells -- a verdict, and a cliff.")
    print(f"  robustness spans {scored['robustness'].min():.3f} to "
          f"{scored['robustness'].max():.3f} -- the gradient a search can climb.")
